In [1]:
from training import TranslationDataset
import torch
import json
from evaluation import SimulMTEvaluator, MTQualityScorer, NLLBSimulMTAdapter

/root/LinearSimultMT/.venv-mamba/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)
nllb = AutoModelForSeq2SeqLM.from_pretrained("./models/nllb_teacher").eval()

In [4]:
adapter = NLLBSimulMTAdapter(
    model=nllb,
    tokenizer=tokenizer,
    name="nllb_wait_k",
    device="cuda",
    use_amp=True
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(
        use_comet=True,
        comet_model_name="Unbabel/wmt22-comet-da",
        comet_batch_size=512,
    ),
)

for k in [3, 5, 7, 9, 11]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=512,
        max_new_tokens=63,
        dataset_fraction=0.02,
        mode="simulmt"
    )
    
    print(result.metrics)
    
    with open(f"./metrics/nllb_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/root/LinearSimultMT/.venv-mamba/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


Validating nllb_wait_k, wait_k=3:   0%|          | 0/13 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/

{'BLEU': 24.737092017213616, 'chrF++': 48.361813373433925, 'TER': 63.91971351518635, 'COMET': 0.8621247221451462, 'AP': 1.0, 'AL': 27.9849709114415, 'DAL': 27.9849709114415, 'LAAL': 15.960123021300307, 'ATD_text': 13.564768754709846, 'total_time_sec': 452.85565478100034, 'ms_per_sentence': 73.182878923885, 'target_tokens_per_sec': 456.227051200042, 'source_tokens_per_sec': 382.39778651708565, 'first_token_latency_sec': 0.33524107533948044, 'peak_gpu_memory_mb': 33835.3310546875, 'dataset_fraction': 0.02, 'eval_dataset_size': 6188, 'full_dataset_size': 309410, 'generation_total_time_sec': 451.78584231899003, 'generation_ms_per_sentence': 73.00999391063188, 'generation_target_tokens_per_sec': 457.3073802833412}


Validating nllb_wait_k, wait_k=5:   0%|          | 0/13 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
index=12
print(result.translations.loc[index, "source"])
print(result.translations.loc[index, "reference"])
print(result.translations.loc[index, "hypothesis"])